# AAC Modelio Apmokymas v2 (100 epochų)
### Šnekutis – pagerintas apmokymas

**Prieš pradedant:** Vykdymo aplinka → Keisti vykdymo aplinkos tipą → **T4 GPU**

---

## 1 žingsnis: Bibliotekų įdiegimas

In [1]:
!pip install transformers torch datasets scikit-learn --quiet

## 2 žingsnis: CSV failo įkėlimas

In [2]:
from google.colab import files
import pandas as pd

print('Įkelk aac_sakiniai.csv failą:')
uploaded = files.upload()

# Ikeliamas csv failas
csv_failas = list(uploaded.keys())[0]
df = pd.read_csv(csv_failas)
print(f'\nFailas įkeltas: {csv_failas}')
print(f'Iš viso sakinių: {len(df)}')

Įkelk aac_sakiniai.csv failą:


Saving aac_sakiniai_v8.csv to aac_sakiniai_v8.csv

Failas įkeltas: aac_sakiniai_v8.csv
Iš viso sakinių: 2131


## 3 žingsnis: Duomenų paruošimas

In [3]:
from sklearn.model_selection import train_test_split

sakiniai = df['sakinys'].tolist()

def paruosti_sakini(sakinys):
    zodziai = sakinys.strip().split()
    if len(zodziai) < 2:
        return None, None
    ivestis = ' '.join(zodziai[:-1]) + ' [MASK]'
    tikslas = zodziai[-1]
    return ivestis, tikslas

ivestys, tikslai = [], []
for s in sakiniai:
    iv, tik = paruosti_sakini(s)
    if iv:
        ivestys.append(iv)
        tikslai.append(tik)

X_train, X_test, y_train, y_test = train_test_split(ivestys, tikslai, test_size=0.1, random_state=42)

print(f'Duomenys paruosti')
print(f'Mokymo sakiniu: {len(X_train)}')
print(f'Testavimo sakiniu: {len(X_test)}')
print(f'\nPavyzdys:')
print(f'  Ivestis:  {X_train[0]}')
print(f'  Tikslas:  {y_train[0]}')

Duomenys paruosti
Mokymo sakiniu: 1917
Testavimo sakiniu: 214

Pavyzdys:
  Ivestis:  mokytoja žaisti kamuolys [MASK]
  Tikslas:  svetainė


## 4 žingsnis: Modelio pakrovimas

In [4]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

MODEL = 'bert-base-multilingual-cased'
print(f'Kraunamas: {MODEL}...')

tokenizer = BertTokenizer.from_pretrained(MODEL)
modelis = BertForMaskedLM.from_pretrained(MODEL)

irenginys = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelis = modelis.to(irenginys)

print(f'\nModelis pakrautas')
print(f'Irenginys: {irenginys}')
if str(irenginys) == 'cpu':
    print('GPU nerastas')
else:
    print('GPU aktyvus')

Kraunamas: bert-base-multilingual-cased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Modelis pakrautas
Irenginys: cuda
GPU aktyvus


## 5 žingsnis: Žodyno kodavimas

In [5]:
import torch

NAUDOTOJO_ZODYNAS = [
    'as', 'tu', 'jis', 'ji',
    'mama', 'tetis', 'draugas', 'mokytoja',
    'suo', 'kate',
    'obuolys', 'sumustinis',
    'vanduo', 'sultys',
    'knyga', 'zaislas', 'kamuolys', 'telefonas',
    'mokykla', 'parduotuve', 'kiemas', 'namai',
    'virtuve', 'svetaine', 'tualetas',
    'eiti',
    'noreti', 'duoti', 'laikyti', 'zaisti', 'piesti',
    'ziureti', 'skaityti',
    'laukti', 'kalbeti', 'taip', 'ne',
    'valgyti', 'gerti', 'miegoti'
]

LIETUVISKAS_ZODYNAS = [
    'aš', 'tu', 'jis', 'ji',
    'mama', 'tėtis', 'draugas', 'mokytoja',
    'šuo', 'katė',
    'obuolys', 'sumuštinis',
    'vanduo', 'sultys',
    'knyga', 'žaislas', 'kamuolys', 'telefonas',
    'mokykla', 'parduotuvė', 'kiemas', 'namai',
    'virtuvė', 'svetainė', 'tualetas',
    'eiti',
    'norėti', 'duoti', 'laikyti', 'žaisti', 'piešti',
    'žiūrėti', 'skaityti',
    'laukti', 'kalbėti', 'taip', 'ne',
    'valgyti', 'gerti', 'miegoti'
]

def uzkoduoti_zodyna(zodynas, tokenizer, modelis, irenginys):
    modelis.eval()
    vektoriai = []
    with torch.no_grad():
        for zodis in zodynas:
            tokenai = tokenizer.tokenize(zodis)
            id_sarasas = tokenizer.convert_tokens_to_ids(tokenai)
            id_tensor = torch.tensor(id_sarasas).to(irenginys)
            embed = modelis.bert.embeddings.word_embeddings(id_tensor)
            vektorius = embed.mean(dim = 0)
            vektoriai.append(vektorius)
    return torch.stack(vektoriai)

print('Koduojamas zodynas...')
zodyno_vektoriai = uzkoduoti_zodyna(LIETUVISKAS_ZODYNAS, tokenizer, modelis, irenginys)
print(f'Zodynas uzkoduotas')
print(f'Zodyno dydis: {len(LIETUVISKAS_ZODYNAS)} korteliu')
print(f'Vektoriaus dimensija: {zodyno_vektoriai.shape[1]}')

Koduojamas zodynas...
Zodynas uzkoduotas
Zodyno dydis: 40 korteliu
Vektoriaus dimensija: 768


## 6 žingsnis: Modelio apmokymas (100 epochų)

In [6]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch import nn
import time

class AACDataset(Dataset):
    def __init__(self, ivestys, tikslai, tokenizer, zodynas):
        self.ivestys = ivestys
        self.tikslai = tikslai
        self.tokenizer = tokenizer
        self.zodynas = zodynas

    def __len__(self):
        return len(self.ivestys)

    def __getitem__(self, idx):
        encoded = self.tokenizer(self.ivestys[idx], return_tensors = 'pt', padding = 'max_length', max_length = 32, truncation = True)
        tikslo_idx = self.zodynas.index(self.tikslai[idx]) if self.tikslai[idx] in self.zodynas else 0
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'tikslas': torch.tensor(tikslo_idx)
        }

train_dataset = AACDataset(X_train, y_train, tokenizer, LIETUVISKAS_ZODYNAS)
test_dataset  = AACDataset(X_test,  y_test,  tokenizer, LIETUVISKAS_ZODYNAS)
train_loader  = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader   = DataLoader(test_dataset,  batch_size = 32)

class AACModelis(nn.Module):
    def __init__(self, bert_modelis, zodyno_vektoriai):
        super().__init__()
        self.bert = bert_modelis.bert
        self.dekoderis = nn.Linear(768, len(LIETUVISKAS_ZODYNAS), bias = False)
        self.dekoderis.weight = nn.Parameter(zodyno_vektoriai.clone())

    def forward(self, input_ids, attention_mask):
        isvestis = self.bert(input_ids = input_ids, attention_mask = attention_mask)
        mask_pozicija = (input_ids == tokenizer.mask_token_id)
        mask_vektoriai = isvestis.last_hidden_state[mask_pozicija]
        logitai = self.dekoderis(mask_vektoriai)
        return logitai

aac_modelis = AACModelis(modelis, zodyno_vektoriai).to(irenginys)

EPOCHOS = 100
optimizer = AdamW(aac_modelis.parameters(), lr = 1e-5, weight_decay = 0.01)
kriterijus = nn.CrossEntropyLoss()

# Mokymosi greicio mazinimas kai modelis nebegereja
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience = 10)

geriausias_nuostolis = float('inf')
geriausias_modelis = None

print('Pradedamas apmokymas (100 epochu)...')
print(f'Mokymo sakiniu: {len(train_dataset)}')
print('-' * 55)

pradzios_laikas = time.time()

for epocha in range(EPOCHOS):
    aac_modelis.train()
    bendras_nuostolis = 0
    teisingi = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(irenginys)
        attention_mask = batch['attention_mask'].to(irenginys)
        tikslai_batch = batch['tikslas'].to(irenginys)

        optimizer.zero_grad()
        logitai = aac_modelis(input_ids, attention_mask)
        nuostolis = kriterijus(logitai, tikslai_batch)
        nuostolis.backward()
        torch.nn.utils.clip_grad_norm_(aac_modelis.parameters(), 1.0)
        optimizer.step()

        bendras_nuostolis += nuostolis.item()
        teisingi += (logitai.argmax(dim = 1) == tikslai_batch).sum().item()

    vid_nuostolis = bendras_nuostolis / len(train_loader)
    tikslumas = teisingi / len(train_dataset) * 100

    # Mokymosi greicio koregavimas
    scheduler.step(vid_nuostolis)

    # Isaugomas geriausias modelis
    if vid_nuostolis < geriausias_nuostolis:
        geriausias_nuostolis = vid_nuostolis
        geriausias_modelis = {k: v.clone() for k, v in aac_modelis.state_dict().items()}

    # Kas 10 epochu
    if (epocha + 1) % 10 == 0:
        praejes = (time.time() - pradzios_laikas) / 60
        print(f'Epocha {epocha+1:3d}/100 | Nuostolis: {vid_nuostolis:.4f} | Tikslumas: {tikslumas:.1f}% | Laikas: {praejes:.1f} min')

# Geriausias modelis
aac_modelis.load_state_dict(geriausias_modelis)
print(f'\nApmokymas baigtas')
print(f'Geriausias nuostolis: {geriausias_nuostolis:.4f}')
print(f'Bendras laikas: {(time.time() - pradzios_laikas) / 60:.1f} min')

Pradedamas apmokymas (100 epochu)...
Mokymo sakiniu: 1917
-------------------------------------------------------
Epocha  10/100 | Nuostolis: 2.0293 | Tikslumas: 17.8% | Laikas: 2.3 min
Epocha  20/100 | Nuostolis: 1.9724 | Tikslumas: 17.3% | Laikas: 4.7 min
Epocha  30/100 | Nuostolis: 1.9415 | Tikslumas: 18.2% | Laikas: 7.0 min
Epocha  40/100 | Nuostolis: 1.9390 | Tikslumas: 17.4% | Laikas: 9.4 min
Epocha  50/100 | Nuostolis: 1.9282 | Tikslumas: 17.9% | Laikas: 11.8 min
Epocha  60/100 | Nuostolis: 1.9083 | Tikslumas: 18.8% | Laikas: 14.1 min
Epocha  70/100 | Nuostolis: 1.9041 | Tikslumas: 17.4% | Laikas: 16.5 min
Epocha  80/100 | Nuostolis: 1.8935 | Tikslumas: 16.9% | Laikas: 18.9 min
Epocha  90/100 | Nuostolis: 1.8832 | Tikslumas: 18.7% | Laikas: 21.2 min
Epocha 100/100 | Nuostolis: 1.8808 | Tikslumas: 18.4% | Laikas: 23.6 min

Apmokymas baigtas
Geriausias nuostolis: 1.8735
Bendras laikas: 23.6 min


## 7 žingsnis: Rankinis Testavimas

In [7]:
def speti_kortele(sakinys, modelis, tokenizer, zodynas, top_k = 5):
    if '[MASK]' not in sakinys:
        sakinys = sakinys + ' [MASK]'
    modelis.eval()
    with torch.no_grad():
        encoded = tokenizer(sakinys, return_tensors = 'pt', padding = 'max_length', max_length = 32, truncation = True)
        input_ids      = encoded['input_ids'].to(irenginys)
        attention_mask = encoded['attention_mask'].to(irenginys)
        logitai = modelis(input_ids, attention_mask)
        tikimybes = torch.softmax(logitai, dim = 1)[0]
        top_indeksai = tikimybes.topk(top_k).indices

    print(f'\nIvestis: "{sakinys}"')
    print(f'Top {top_k} siulomos korteles:')
    for i, idx in enumerate(top_indeksai):
        print(f'  {i+1}. {zodynas[idx]:20s} ({tikimybes[idx]*100:.1f}%)')

print('=' * 50)
print('MODELIO TESTAVIMAS')
print('=' * 50)

testai = [
    'aš norėti',
    'aš valgyti',
    'aš gerti',
    'aš eiti',
    'mama duoti',
    'tu žaisti',
    'aš norėti valgyti',
    'aš eiti mokykla',
    'šuo eiti',
]

for testas in testai:
    speti_kortele(testas, aac_modelis, tokenizer, LIETUVISKAS_ZODYNAS)
    print()

MODELIO TESTAVIMAS

Ivestis: "aš norėti [MASK]"
Top 5 siulomos korteles:
  1. žaislas              (14.1%)
  2. vanduo               (13.9%)
  3. kamuolys             (13.4%)
  4. knyga                (12.8%)
  5. sultys               (12.3%)


Ivestis: "aš valgyti [MASK]"
Top 5 siulomos korteles:
  1. obuolys              (63.3%)
  2. sumuštinis           (35.5%)
  3. kamuolys             (0.4%)
  4. vanduo               (0.1%)
  5. žaislas              (0.1%)


Ivestis: "aš gerti [MASK]"
Top 5 siulomos korteles:
  1. vanduo               (49.8%)
  2. sultys               (49.7%)
  3. sumuštinis           (0.1%)
  4. obuolys              (0.1%)
  5. šuo                  (0.1%)


Ivestis: "aš eiti [MASK]"
Top 5 siulomos korteles:
  1. virtuvė              (16.7%)
  2. svetainė             (15.3%)
  3. tualetas             (15.1%)
  4. mokykla              (15.0%)
  5. parduotuvė           (14.8%)


Ivestis: "mama duoti [MASK]"
Top 5 siulomos korteles:
  1. vanduo               (20.3%)


In [8]:
##Atsitiktinių sakinių testavimas

import random

def speti_kortele_rezultatas(sakinys, modelis, tokenizer, zodynas, top_k=5):
    if '[MASK]' not in sakinys:
        sakinys = sakinys + ' [MASK]'
    modelis.eval()
    with torch.no_grad():
        encoded = tokenizer(sakinys, return_tensors='pt',
                           padding='max_length', max_length=32, truncation=True)
        input_ids      = encoded['input_ids'].to(irenginys)
        attention_mask = encoded['attention_mask'].to(irenginys)
        logitai = modelis(input_ids, attention_mask)
        tikimybes = torch.softmax(logitai, dim=1)[0]
        top_indeksai = tikimybes.topk(top_k).indices.tolist()
        return [(zodynas[i], round(tikimybes[i].item() * 100, 1)) for i in top_indeksai]

print("=" * 55)
print("ATSITIKTINIŲ SAKINIŲ TESTAVIMAS (10 sakiniu)")
print("=" * 55)

atsitiktiniai = random.sample(list(zip(X_test, y_test)), 10)

teisingi_top1 = 0
teisingi_top3 = 0

for sakinys, teisingas in atsitiktiniai:
    tekstas = sakinys.replace(' [MASK]', '')
    siulymai = speti_kortele_rezultatas(tekstas, aac_modelis, tokenizer, LIETUVISKAS_ZODYNAS)

    print(f'\nĮvestis: "{tekstas} [MASK]"')
    print(f'Top 5 siulomos korteles:')
    for i, (zodis, tikimybe) in enumerate(siulymai):
        print(f'     {i+1}. {zodis:20s} ({tikimybe}%)')

ATSITIKTINIŲ SAKINIŲ TESTAVIMAS (10 sakiniu)

Įvestis: "aš kalbėti tėtis [MASK]"
Top 5 siulomos korteles:
     1. parduotuvė           (47.5%)
     2. namai                (25.2%)
     3. kiemas               (15.6%)
     4. mokykla              (8.3%)
     5. virtuvė              (0.4%)

Įvestis: "mama duoti draugas [MASK]"
Top 5 siulomos korteles:
     1. žaislas              (15.4%)
     2. kamuolys             (14.5%)
     3. sumuštinis           (12.4%)
     4. telefonas            (12.4%)
     5. obuolys              (12.2%)

Įvestis: "ji duoti draugas [MASK]"
Top 5 siulomos korteles:
     1. sumuštinis           (17.0%)
     2. obuolys              (16.3%)
     3. vanduo               (14.7%)
     4. sultys               (13.6%)
     5. kamuolys             (11.9%)

Įvestis: "mama žaisti kamuolys [MASK]"
Top 5 siulomos korteles:
     1. mokykla              (54.4%)
     2. svetainė             (42.9%)
     3. kiemas               (1.0%)
     4. namai                (0.6%)
     5

## 8 žingsnis: Testavimo tikslumas

In [9]:
aac_modelis.eval()
teisingi = 0
top3_teisingi = 0
viso = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(irenginys)
        attention_mask = batch['attention_mask'].to(irenginys)
        tikslai_batch  = batch['tikslas'].to(irenginys)

        logitai = aac_modelis(input_ids, attention_mask)
        tikimybes = torch.softmax(logitai, dim = 1)

        # Top-1 tikslumas
        teisingi += (logitai.argmax(dim=1) == tikslai_batch).sum().item()

        # Top-3 tikslumas
        top3 = tikimybes.topk(3).indices
        for i, tikslas in enumerate(tikslai_batch):
            if tikslas in top3[i]:
                top3_teisingi += 1
        viso += len(tikslai_batch)


print(f'Top-1 tikslumas: {teisingi/viso*100:.1f}%')
print(f'Top-3 tikslumas: {top3_teisingi/viso*100:.1f}%')
print(f'Testavimo sakinių: {viso}')

Top-1 tikslumas: 2.8%
Top-3 tikslumas: 16.8%
Testavimo sakinių: 214


## 9 žingsnis: Modelio išsaugojimas

In [10]:
import os, json, shutil
from google.colab import files

os.makedirs('aac_modelis', exist_ok = True)
torch.save(aac_modelis.state_dict(), 'aac_modelis/modelio_svoriai.pt')
tokenizer.save_pretrained('aac_modelis/')

with open('aac_modelis/zodynas.json', 'w', encoding = 'utf-8') as f:
    json.dump(LIETUVISKAS_ZODYNAS, f, ensure_ascii = False, indent = 2)

shutil.make_archive('aac_modelis', 'zip', 'aac_modelis')
files.download('aac_modelis.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>